In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# PT-W2-D4：Rule 抽取 —— 业务规则在代码里还是在 Ontology 里？

> 📅 Week 2 - Day 4（2026-08-06 周三）
>
> **并行轨道：Business Semantic Architecture**
>
> **昨日回顾**：D3 从 effect-registry 抽取了 Lifecycle + Event，发现 5 类冻结 effect_type 覆盖了状态传播链，但完整状态机仍有缺口。今天进入 Rule 维度。

---


## 今日主题：Rule 抽取

### 核心问题

> **你的业务规则，AI 能读到吗？**

一条业务规则在 ERP 系统里有三种存在形态：


In [ ]:
形态 1：显式声明在 Ontology / BCM 矩阵中
         → AI 能直接读取并推理 ✅

形态 2：写在代码的 if-else 分支里
         → AI 看不到，必须读源代码才能理解 ❌

形态 3：存在于业务人员的头脑中，从未被文档化
         → AI 完全无法感知 💀


**Ontology 层的 Rule 抽取，就是把形态 2 和形态 3 的规则提升到形态 1。**

---


## 一、什么是 Business Rule？

在 Ontology 语境下，Business Rule 不是"验证规则"（如"手机号 11 位"），而是：

> **决定业务实体在什么条件下可以（或必须）发生什么状态变化的约束。**

对照你已有的 BCM 域文件，Rule 回答的是这类问题：

| 问题 | 来自哪个域 | 你现在的答案在哪里 |
|------|-----------|-----------------|
| "非标报价必须升级审批" | 01 招商 | BCM 行 CRE-SAL-021 业务规则列 |
| "资产性铺位变更须 K2 审批" | 03 租赁 | BCM 行 CRE-LEA-003 业务规则列 |
| "已生成账单的合同不可删除，只能终止" | 02 合同 | BCM 行 CRE-CON-001 业务规则列 |
| "核销优先级：账款→预存款→保证金→意向金" | 04 财务 | BCM 行 CRE-FIN-015 业务规则列 |
| "合同终止后铺位状态变更" | 跨域 | effect-registry: occupancy-effect |

---


## 二、从你的 BCM 域文件中抽取 Rule

### 2.1 Rule 的三种类型

通过阅读你的 01-招商管理、02-合同管理、03-租赁管理、04-财务管理四个域文件，可以把规则分为三类：

#### 类型 A：**校验规则（Validation Rule）**

> 在数据写入前校验业务合法性。

| 来源 | 规则 | AI 可读性 |
|------|------|----------|
| CRE-SAL-001 | 中文名与外文名联合校验唯一 | ✅ BCM 显式声明 |
| CRE-SAL-007 | 购物中心普通合同只能选择租户已授权品牌 | ✅ BCM 显式声明 |
| CRE-LEA-007 | 状态转换校验阻止非法跃迁 | ⚠️ BCM 声明但七态/四态/六态分歧未统一（G1） |
| CRE-FIN-002 | 已终止合同零账单（结算配置参数） | ✅ BCM 显式声明 |

**特征**：这类规则最适合被 AI Agent 在**执行前做预校验**——"我要创建一条报价，但品牌未授权，所以不能继续"。

#### 类型 B：**路由规则（Routing Rule）**

> 决定业务流向和审批路径。

| 来源 | 规则 | AI 可读性 |
|------|------|----------|
| CRE-SAL-021 | 判标非标 → 升级审批 | ✅ BCM 显式声明 |
| CRE-SAL-019 | 超出授权底线 → 升级审批 | ✅ BCM 显式声明 |
| CRE-LEA-003 | 非资产性铺位 → 直接保存；资产性铺位 → K2 审批 | ✅ BCM 显式声明 |
| CRE-FIN-015 | 核销优先级：1 账款 → 2 预存款 → 3 保证金 → 4 意向金 | ✅ BCM 显式声明 |

**特征**：这类规则是 AI Agent 做**任务规划**的依据——"收到一个审批请求，我应该走哪条路径？"

#### 类型 C：**推理规则（Inference Rule）**

> 需要跨域推理、综合多条件才能得出的结论。

| 来源 | 规则 | AI 可读性 |
|------|------|----------|
| CRE-SAL-026→CRE-LEA-008 | 预定生效 → 资源状态变为 locked | ⚠️ 需要 ADR-006 effect 声明才能推理 |
| CRE-CON-018→CRE-LEA-007 | 合同审批通过 → 铺位变为使用中 | ⚠️ 需要 PRD §3.2 L1934 跨模块联动表 |
| CRE-CON-024→CRE-FIN-008 | 合同终止 → 后续账单应停止（零账单） | ⚠️ 需要 04 结算配置 + 02 终止逻辑联合推理 |
| CRE-SAL-021→CRE-CON-018 | 报价判标通过 → 可转意向 → 可转合同 | ⚠️ 需要 effect-registry: state-transition-effect + lead-conversion-effect |

**特征**：这类规则是 AI Agent 做**业务链推理**的核心——"A101 铺位为什么不能出租？"需要沿这条链推导。

**推理规则是你最大的缺口。** 单看一个域的 BCM 能理解"本域规则"，但跨域推理依赖的是 effect-registry 和 ADR-006 的 Lifecycle Transition Effect——目前只有 5 个 effect_type，远未覆盖所有跨域影响。

---


## 三、实战抽取：从 BCM 文件中提取 Rule 清单

### 3.1 抽取方法


In [ ]:
Step 1：打开 BCM 域文件
Step 2：定位 §3 能力矩阵主表中的"业务规则"列
Step 3：对每条规则做分类（校验/路由/推理）
Step 4：检查该规则是否跨域（是否依赖其他域的状态变化）
Step 5：如果是推理规则，检查 effect-registry 中是否有对应声明


### 3.2 实例：从 01-招商管理抽取 Rule

以 CRE-SAL-021（招商政策判标）为例：


In [ ]:
原文：
"租金、租期、免租、保证金、递增、品牌级别逐项判定；
 达标绿色、非标红色；
 非标升级审批"

Rule 抽取结果：
┌──────────────────────────────────────────────┐
│ Rule ID: RULE-SAL-021-01                     │
│ 类型: 校验规则                                │
│ 条件: 报价提交时                              │
│ 校验项: 租金 ≥ 一铺一价底价                   │
│         租期 ∈ 政策范围                       │
│         免租期 ≤ 政策上限                     │
│         保证金 ≥ 政策最低                     │
│         递增率 ≥ 政策基准                     │
│         品牌级别 ∈ 允许级别                   │
│ 结果: 全部达标 → 绿色（通过）                  │
│       任一非标 → 红色（升级审批）               │
│ 跨域影响: 判标通过 → 可转意向 (lead-conversion) │
│ AI 可读性: ✅ BCM 显式声明                     │
└──────────────────────────────────────────────┘


### 3.3 实例：从 04-财务管理抽取 Rule

以 CRE-FIN-015（收款单与核销）为例：


In [ ]:
原文：
"核销优先级：1 账款、2 预存款冲抵、3 保证金冲抵、4 意向金冲抵"

Rule 抽取结果：
┌──────────────────────────────────────────────┐
│ Rule ID: RULE-FIN-015-01                     │
│ 类型: 路由规则                                │
│ 条件: 收款生效时                              │
│ 执行顺序:                                    │
│   1. 先核销账款                              │
│   2. 不足 → 预存款冲抵                        │
│   3. 不足 → 保证金冲抵                        │
│   4. 不足 → 意向金冲抵                        │
│   5. 仍有余 → 转预存款（长短款处理）            │
│ 跨域影响: 保证金冲抵影响 02 合同的保证金余额     │
│ AI 可读性: ✅ BCM 显式声明                     │
│ 关联: CRE-FIN-016 长短款处理                   │
└──────────────────────────────────────────────┘


### 3.4 发现缺口：代码里的隐式规则

对照 BCM 域文件中的 Review（§6）待确认事项，可以发现**规则虽然写了但实现细节仍藏在代码里**的情况：

| BCM Review 项 | 隐式规则 | 为什么 AI 读不到 |
|---------------|---------|----------------|
| G2（03 租赁）| 资源占用率 ≠ 出租率 | `resources.status='occupied'` 和 `lease_contracts.status='effective'` 是 SQL 过滤条件，不是声明式规则 |
| G4（03 租赁）| 招商锁定铺位的业务路径 | `locked` 状态在代码里通过直接 SQL 建立，无公开 API 入口 |
| G7（03 租赁）| 所有 MI 证据为隔离测试级 | 状态转换校验只在代码的 service 层 if-else，无声明式状态机 |
| 保底取高算法（04 财务）| 保底 vs 提成取高的周期模式 | 自然月/合同月/固定日的取法在算法代码中，BCM 只说"取高" |

---


## 四、Ontology 视角：Rule 应该怎么表达？

### 4.1 面向 AI 的 Rule 声明格式

一条 AI 可推理的 Rule 至少需要四个要素：


In [ ]:
Rule: <唯一 ID>
When: <触发条件>（事件 + 前置状态）
Check: <校验逻辑>（可分解的判断条件）
Then: <结果>（通过→下一步 / 不通过→拒绝或升级）
Evidence: <关联的 BCM 行 / ADR / PRD 章节>


### 4.2 对照你已有的资产

| 要素 | 你已有什么 | 缺什么 |
|------|-----------|--------|
| When | ✅ effect-registry 的 BusinessEventType | 状态机不完整 |
| Check | ⚠️ BCM 业务规则列（自然语言描述） | 未结构化为可执行判断条件 |
| Then | ⚠️ BCM 业务规则列有文字描述 | 缺与 Capability 的映射（通过后调哪个 API？） |
| Evidence | ✅ BCM §5 证据可追溯性 | 充分 |

### 4.3 关键洞察：你的 BCM 已经是半结构化 Rule 库

**重要发现：你的 BCM 矩阵的"业务规则"列，本质上已经是一个 Rule Catalog。** 每一行 BCM 的"业务规则"单元格，包含了一条或多条规则的自然语言描述。

缺的不是规则本身，而是：

1. **结构化**：从自然语言 → If-Then 格式
2. **分类标注**：区分校验规则 / 路由规则 / 推理规则
3. **跨域链接**：标注哪些规则依赖 effect-registry 的跨域声明
4. **AI 可执行映射**：通过后应该调用哪个 Capability（BCM 行 ID）

---


## 五、与主线（Governance）的连接

### 架构师视角


In [ ]:
以前：业务规则 = 代码里的 if-else
      改规则 = 改代码 = 需要开发周期

现在：业务规则 = Ontology Rule 声明
      改规则 = 改声明 = 配置层变更
      AI Agent 可以在运行时读取规则做推理


### 与主线 Week 10（Governance）的关联

| 主线概念 | 并行轨道对应 |
|---------|------------|
| Policy（治理策略） | Rule 的上层包装——Policy = Rule + 授权范围 |
| Audit（审计） | Rule 执行日志 = 审计证据 |
| Fail-closed | Rule 不满足时的默认行为 |

主线关注"谁有权做什么"（Policy），并行轨道关注"什么条件下可以做什么"（Rule）。两者结合 = 完整的 Agent 执行约束。

---


## 六、练习（5 分钟）

**选择你 BCM 中的一条推理规则，尝试结构化。**

推荐选择：CRE-CON-024（合同终止）→ CRE-LEA-007（铺位状态）→ CRE-FIN-008（零账单）

问题链：
1. 合同终止时，铺位状态应该怎么变？（查 effect-registry: occupancy-effect）
2. 合同终止后，已生成的未来账单怎么处理？（查 CRE-FIN-002 结算配置：已终止合同零账单）
3. 合同终止后，如果退场检查未完成，铺位能立即变为"可经营"吗？（查 CRE-LEA-011 交付与进退场）

请用上面的四要素格式（When / Check / Then / Evidence）写出来。

---


## 📊 本日 Rule 抽取进度

| Ontology 维度 | 已有资产 | 今天抽取发现 | 缺口 |
|--------------|---------|------------|------|
| **Rule** | BCM 业务规则列（自然语言） | 三类规则（校验/路由/推理）；BCM 已是半结构化 Rule 库 | 推理规则缺结构化；跨域规则缺 effect 声明；代码隐式规则需提升 |

---


## 明日预告

**D5：Capability 抽取** —— BCM capability 行 → LangChat Capability 映射。

你的 BCM 每一行就是一个 Capability，但它们和 LangChat 的 Capability 概念是否对齐？明天对照 capability-traceability-matrix.md。
